# Overview
The goal is to automatize the conversion of single-patient data in .xlsx format from a single folder into a unique .xlsx file with some variables of interest.
I'd work as follow:
1. List variables in single-patient files and define the ones to keep to be put in the final file.
2. Create a dictionary with these latters and a variable saying wheather more-than-one obs. for each variable should be keepen or not.
3. Create a dictionary with variables for the final file, a variable explaining each of them and a variable with the format for each one
4. Write a functions that:
    - Automatically goes into data folder
    - Reads file by file and, for each of them:
        - Keeps variables of interest
        - Renames and reformat them after final file dictionary
        - For text variables, it is able to find relevant informations (e.g. drug names froma full prescription paragraph)


# Preliminary operations

## Variables listing

### Row files

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


def list_spreadsheet_files(data_folder):
    """
    List .xlsx, .xls and .ods files in the data folder.
    """
    data_folder = Path(data_folder)

    extensions = ["*.xlsx", "*.xls", "*.ods"]

    files = []
    for ext in extensions:
        files.extend(data_folder.glob(ext))

    files = sorted(files)

    if len(files) == 0:
        raise FileNotFoundError(
            f"No .xlsx, .xls or .ods files found in folder: {data_folder}"
        )

    return files


def get_excel_engine(file_path):
    """
    Select the correct pandas engine according to file extension.
    """
    suffix = Path(file_path).suffix.lower()

    if suffix == ".xlsx":
        return "openpyxl"
    elif suffix == ".xls":
        return "xlrd"
    elif suffix == ".ods":
        return "odf"
    else:
        raise ValueError(f"Unsupported file extension: {suffix}")


def classify_variable_type(values, paragraph_word_threshold=12, paragraph_char_threshold=80):
    """
    Classify a variable as:
    - empty
    - paragraph
    - unique_value

    A variable is classified as paragraph if at least one non-empty cell has:
    - a line break, or
    - at least paragraph_word_threshold words, or
    - at least paragraph_char_threshold characters.
    """

    values = values.dropna()

    if values.empty:
        return "empty"

    for value in values:
        text = str(value).strip()

        if text == "":
            continue

        word_count = len(text.split())
        char_count = len(text)

        has_line_break = "\n" in text or "\r" in text

        if (
            has_line_break
            or word_count >= paragraph_word_threshold
            or char_count >= paragraph_char_threshold
        ):
            return "paragraph"

    return "unique_value"


def list_variables_from_patient_files(
    data_folder="row_data",
    output_path=None,
    paragraph_word_threshold=12,
    paragraph_char_threshold=80
):
    """
    Lists all different variable names appearing in spreadsheet files.

    Accepted formats:
    - .xlsx
    - .xls
    - .ods

    Variable names are assumed to be in row 2 of each sheet.
    Each sheet of each file is treated as one independent patient-file unit.

    Adds variable_type:
    - unique_value
    - paragraph
    - empty
    """

    files = list_spreadsheet_files(data_folder)

    records = []

    for file_path in files:

        engine = get_excel_engine(file_path)

        try:
            spreadsheet_file = pd.ExcelFile(file_path, engine=engine)
        except Exception as e:
            print(f"Could not read file: {file_path.name}. Error: {e}")
            continue

        for sheet_name in spreadsheet_file.sheet_names:

            try:
                df = pd.read_excel(
                    file_path,
                    sheet_name=sheet_name,
                    header=1,
                    engine=engine
                )

                # Clean variable names
                df.columns = [
                    str(col).strip() if not pd.isna(col) else ""
                    for col in df.columns
                ]

                # Remove empty column names
                df = df.loc[:, df.columns != ""]

                for variable in df.columns:
                    variable_type = classify_variable_type(
                        df[variable],
                        paragraph_word_threshold=paragraph_word_threshold,
                        paragraph_char_threshold=paragraph_char_threshold
                    )

                    records.append({
                        "file_name": file_path.name,
                        "file_type": file_path.suffix.lower(),
                        "sheet_name": sheet_name,
                        "patient_sheet_id": f"{file_path.name} | {sheet_name}",
                        "variable_name": variable,
                        "variable_type": variable_type
                    })

            except Exception as e:
                print(
                    f"Could not read sheet '{sheet_name}' "
                    f"in file '{file_path.name}'. Error: {e}"
                )

    file_sheet_variables = pd.DataFrame(records)

    if file_sheet_variables.empty:
        raise ValueError("No variables were found in the spreadsheet files.")

    total_patient_sheets = file_sheet_variables["patient_sheet_id"].nunique()

    variable_summary = (
        file_sheet_variables
        .drop_duplicates(["patient_sheet_id", "variable_name"])
        .groupby("variable_name")
        .agg(
            n_files=("patient_sheet_id", "nunique"),
            variable_type=(
                "variable_type",
                lambda x: "paragraph" if "paragraph" in list(x)
                else "unique_value" if "unique_value" in list(x)
                else "empty"
            )
        )
        .reset_index()
    )

    variable_summary["percentage_files"] = (
        variable_summary["n_files"] / total_patient_sheets * 100
    ).round(1)

    variable_summary = variable_summary.sort_values(
        by=["n_files", "variable_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            variable_summary.to_excel(
                writer,
                sheet_name="variable_summary",
                index=False
            )

            file_sheet_variables.to_excel(
                writer,
                sheet_name="file_sheet_variables",
                index=False
            )

    return variable_summary, file_sheet_variables

list_variables_from_patient_files(data_folder="dati/row_data", output_path="dati/variable_summary.xlsx")

FileNotFoundError: No .xlsx, .xls or .ods files found in folder: dati\row_data

### Final file

In [ ]:
from pathlib import Path
import pandas as pd


def get_excel_engine(file_path):
    """
    Select the correct pandas engine according to file extension.
    """
    suffix = Path(file_path).suffix.lower()

    if suffix == ".xlsx":
        return "openpyxl"
    elif suffix == ".xls":
        return "xlrd"
    elif suffix == ".ods":
        return "odf"
    else:
        raise ValueError(f"Unsupported file extension: {suffix}")


def list_reference_variables(reference_file, output_path=None):
    """
    List all variables from the final-result reference file.

    Rules
    -----
    - sheet_name identifies the variable group.
    - For most sheets:
        * row 1 contains subgroup names
        * row 2 contains variable names
        * merged cells in row 1 are handled by forward filling
    - For exception sheets:
        * COMORBIDITA: variable names in row 1, columns A:E only
        * INTERAZIONI: variable names in row 1, columns A:K only
        * ADR: variable names in row 1, all columns
        * subgroup_name is set to missing

    Parameters
    ----------
    reference_file : str
        Path to the reference file.

    output_path : str or None
        If provided, saves the result as an Excel file.

    Returns
    -------
    reference_variables : pandas.DataFrame
        Table with sheet_name, subgroup_name and variable_name.
    """

    reference_file = Path(reference_file)
    engine = get_excel_engine(reference_file)

    spreadsheet = pd.ExcelFile(reference_file, engine=engine)

    records = []

    exception_sheets = {
        "COMORBIDITA": {"header_row": 0, "usecols": "A:E"},
        "INTERAZIONI": {"header_row": 0, "usecols": "A:K"},
        "ADR": {"header_row": 0, "usecols": None}
    }

    for sheet_name in spreadsheet.sheet_names:

        sheet_name_clean = str(sheet_name).strip()
        sheet_name_upper = sheet_name_clean.upper()

        if sheet_name_upper in exception_sheets:

            rule = exception_sheets[sheet_name_upper]

            try:
                df_header = pd.read_excel(
                    reference_file,
                    sheet_name=sheet_name,
                    header=rule["header_row"],
                    nrows=0,
                    usecols=rule["usecols"],
                    engine=engine
                )

            except Exception as e:
                print(f"Could not read sheet '{sheet_name}'. Error: {e}")
                continue

            variables = [
                str(col).strip()
                for col in df_header.columns
                if not pd.isna(col) and str(col).strip() != ""
            ]

            for variable_name in variables:
                records.append({
                    "sheet_name": sheet_name_clean,
                    "subgroup_name": None,
                    "variable_name": variable_name
                })

        else:

            try:
                raw_header = pd.read_excel(
                    reference_file,
                    sheet_name=sheet_name,
                    header=None,
                    nrows=2,
                    engine=engine
                )

            except Exception as e:
                print(f"Could not read sheet '{sheet_name}'. Error: {e}")
                continue

            if raw_header.shape[0] < 2:
                print(f"Sheet '{sheet_name}' skipped: fewer than 2 rows.")
                continue

            subgroup_row = raw_header.iloc[0].ffill()
            variable_row = raw_header.iloc[1]

            for col_idx, variable_name in enumerate(variable_row):

                if pd.isna(variable_name) or str(variable_name).strip() == "":
                    continue

                subgroup_name = subgroup_row.iloc[col_idx]

                if pd.isna(subgroup_name) or str(subgroup_name).strip() == "":
                    subgroup_name = None
                else:
                    subgroup_name = str(subgroup_name).strip()

                records.append({
                    "sheet_name": sheet_name_clean,
                    "subgroup_name": subgroup_name,
                    "variable_name": str(variable_name).strip()
                })

    reference_variables = pd.DataFrame(records)

    if reference_variables.empty:
        raise ValueError("No variables were found in the reference file.")

    reference_variables = reference_variables.sort_values(
        by=["sheet_name", "subgroup_name", "variable_name"],
        na_position="last"
    ).reset_index(drop=True)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            reference_variables.to_excel(
                writer,
                sheet_name="reference_variables",
                index=False
            )

    return reference_variables


list_reference_variables(reference_file="dati/std_configuration.xlsx", output_path="dati/reference_variables.xlsx")

,sheet_name,subgroup_name,variable_name
0,ADR,None,APPLICAZIONE ALGORITMO DI DIPS
1,ADR,None,DATA INSERIMENTO IN RNF
2,ADR,None,GRAVITA'
3,ADR,None,ID PAZIENTE
4,ADR,None,N° SCHEDA ADR
...,...,...,...
118,TERAPIA PRE MR E MEDICATION REV,PRE MR,VIA DI SOMMINISTRAZIONE
119,TERAPIA PRE MR E MEDICATION REV,RIVALUTAZIONE DOPO COLLOQUIO CON IL MEDICO,COMMENTO
120,TERAPIA PRE MR E MEDICATION REV,RIVALUTAZIONE DOPO COLLOQUIO CON IL MEDICO,MODIFICA 1_ACCETTATA
121,TERAPIA PRE MR E MEDICATION REV,RIVALUTAZIONE DOPO COLLOQUIO CON IL MEDICO,MODIFICA 2_ ACCETTATA


# Conversion operations

### Preliminary operations

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# Paths

LINK_PATH = Path("dati/link_variabili_grezze_e_finali.xlsx")
RAW_PATIENT_PATH = Path("dati/row_data/pz_nefro_example.xls")

OUTPUT_PATH = Path("dati/output/pz_nefro_example_mapped_ALL.xlsx")
LOG_PATH = Path("dati/output/pz_nefro_example_mapped_ALL_log.xlsx")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Helpers

def get_engine(path):
    """
    Select the correct pandas engine according to file extension.
    """
    suffix = Path(path).suffix.lower()

    if suffix == ".xlsx":
        return "openpyxl"
    elif suffix == ".xls":
        return "xlrd"
    elif suffix == ".ods":
        return "odf"
    else:
        raise ValueError(f"Unsupported file extension: {suffix}")


def clean_string(x):
    """
    Clean strings without destroying meaningful internal spaces.
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x == "":
        return None

    return x


def normalize_name(x):
    """
    Normalize names for matching.

    This removes leading/trailing spaces, collapses repeated spaces,
    and converts to uppercase.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip()
    x = " ".join(x.split())
    x = x.upper()

    return x


def make_unique_columns(columns):
    """
    Make duplicate column names unique.

    Example:
    A, A, B -> A, A__2, B
    """
    seen = {}
    out = []

    for col in columns:
        col = clean_string(col)

        if col is None:
            col = "Unnamed"

        if col not in seen:
            seen[col] = 1
            out.append(col)
        else:
            seen[col] += 1
            out.append(f"{col}__{seen[col]}")

    return out


def collapse_values(values):
    """
    Collapse values from a raw source column into a single cell.

    For now:
    - remove missing values
    - convert to string
    - strip spaces
    - remove empty strings
    - join multiple values with '; '

    Later we can replace this with variable-specific extraction rules.
    """
    if values is None:
        return np.nan

    cleaned = []

    for value in values:
        if pd.isna(value):
            continue

        value = str(value).strip()

        if value == "":
            continue

        cleaned.append(value)

    if len(cleaned) == 0:
        return np.nan

    return "; ".join(cleaned)

# Read LINK file

def read_link_table(link_path, sheet_name="LINK"):
    """
    Read the mapping table.

    Required columns for this step:
    - new_name
    - short_name
    - source

    Other columns, such as sheet_name, group_name and old_name,
    are allowed but not required for the current output logic.
    """
    engine = get_engine(link_path)

    link = pd.read_excel(
        link_path,
        sheet_name=sheet_name,
        engine=engine
    )

    required_cols = ["new_name", "short_name", "source"]

    missing = [col for col in required_cols if col not in link.columns]

    if len(missing) > 0:
        raise ValueError(f"Missing required columns in LINK sheet: {missing}")

    for col in link.columns:
        link[col] = link[col].apply(clean_string)

    # Keep only rows that define an output variable
    link = link[
        link["new_name"].notna() &
        link["short_name"].notna()
    ].copy()

    return link.reset_index(drop=True)

# Read raw patient workbook

def read_raw_patient_sheets(raw_patient_path):
    """
    Read all sheets from the raw patient workbook.

    Each sheet is treated as one patient.

    Assumption:
    - raw variable names are in row 2, so header=1.
    """
    engine = get_engine(raw_patient_path)
    xls = pd.ExcelFile(raw_patient_path, engine=engine)

    raw_sheets = {}

    for sheet_name in xls.sheet_names:

        df = pd.read_excel(
            raw_patient_path,
            sheet_name=sheet_name,
            header=1,
            engine=engine
        )

        df.columns = make_unique_columns(df.columns)

        raw_sheets[sheet_name] = df

    return raw_sheets


def find_source_column(patient_df, source_name):
    """
    Find a source column inside one patient sheet.

    Matching is case-insensitive and ignores extra spaces.

    Returns:
    - matched column name
    - matched column values
    """
    source_norm = normalize_name(source_name)

    if source_norm == "":
        return None, None

    column_lookup = {
        normalize_name(col): col
        for col in patient_df.columns
    }

    if source_norm not in column_lookup:
        return None, None

    matched_col = column_lookup[source_norm]

    return matched_col, patient_df[matched_col]

# Main function

def build_all_dataset_from_link(
    link_path,
    raw_patient_path,
    output_path,
    log_path=None
):
    """
    Build final ALL-sheet dataset using only:

    - LINK sheet from link_variabili_grezze_e_finali.xlsx
    - raw patient workbook pz_nefro_example.xls

    Output structure:
    - one sheet called ALL
    - row 1: Id + new_name values
    - row 2: id + short_name values
    - rows 3 onward: one row per patient sheet
    """

    link = read_link_table(link_path, sheet_name="LINK")
    raw_sheets = read_raw_patient_sheets(raw_patient_path)

    # ------------------------------------------------------------
    # Output headers from LINK
    # ------------------------------------------------------------

    output_new_names = ["Id"] + link["new_name"].tolist()
    output_short_names = ["id"] + link["short_name"].tolist()

    output_rows = []
    log_records = []

    # ------------------------------------------------------------
    # One row per patient sheet
    # ------------------------------------------------------------

    for patient_id, (patient_sheet_name, patient_df) in enumerate(raw_sheets.items(), start=1):

        patient_row = [patient_id]

        for link_row_idx, link_row in link.iterrows():

            new_name = link_row["new_name"]
            short_name = link_row["short_name"]
            source = link_row["source"]

            # Case 1: source is empty in LINK
            if source is None:
                patient_row.append(np.nan)

                log_records.append({
                    "patient_id": patient_id,
                    "patient_sheet_name": patient_sheet_name,
                    "new_name": new_name,
                    "short_name": short_name,
                    "source": source,
                    "raw_column_found": None,
                    "status": "empty_source_in_LINK"
                })

                continue

            # Case 2: source exists, look for it in current patient sheet
            raw_column_found, raw_values = find_source_column(
                patient_df=patient_df,
                source_name=source
            )

            # Case 3: source not found in raw patient sheet
            if raw_column_found is None:
                patient_row.append(np.nan)

                log_records.append({
                    "patient_id": patient_id,
                    "patient_sheet_name": patient_sheet_name,
                    "new_name": new_name,
                    "short_name": short_name,
                    "source": source,
                    "raw_column_found": None,
                    "status": "source_not_found_in_patient_sheet"
                })

                continue

            # Case 4: source found, extract values
            extracted_value = collapse_values(raw_values)

            patient_row.append(extracted_value)

            if pd.isna(extracted_value):
                status = "source_found_but_empty"
            else:
                status = "mapped"

            log_records.append({
                "patient_id": patient_id,
                "patient_sheet_name": patient_sheet_name,
                "new_name": new_name,
                "short_name": short_name,
                "source": source,
                "raw_column_found": raw_column_found,
                "status": status
            })

        output_rows.append(patient_row)

    # ------------------------------------------------------------
    # Build ALL sheet
    # ------------------------------------------------------------

    all_sheet_df = pd.DataFrame(
        [output_new_names, output_short_names] + output_rows
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        all_sheet_df.to_excel(
            writer,
            sheet_name="ALL",
            index=False,
            header=False
        )

    # ------------------------------------------------------------
    # Build log
    # ------------------------------------------------------------

    log_df = pd.DataFrame(log_records)

    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        with pd.ExcelWriter(log_path, engine="openpyxl") as writer:
            log_df.to_excel(
                writer,
                sheet_name="mapping_log",
                index=False
            )

    return all_sheet_df, log_df


In [ ]:
all_sheet_df, log_df = build_all_dataset_from_link(
    link_path=LINK_PATH,
    raw_patient_path=RAW_PATIENT_PATH,
    output_path=OUTPUT_PATH,
    log_path=LOG_PATH
)

log_df["status"].value_counts()

status
mapped                               171
source_found_but_empty                85
empty_source_in_LINK                  57
source_not_found_in_patient_sheet     29
Name: count, dtype: int64

In [ ]:
all_sheet_df.head()

,0,1,2,3,4,5,6,7,8,9,...,105,106,107,108,109,110,111,112,113,114
0,Id,Comorbidita Pre MR,Data MR,ACB Score Post MR,N. Farmaci Inappropriati Post MR (Criteri Di ...,N. Farmaci Inappropriati Post MR (Start),N. Farmaci Inappropriati Post MR (Stopp),Numero Interazioni Post MR C-D,ACB Score Pre MR,N. Farmaci Inappropriati Pre MR (Criteri Di B...,...,N. Farmaci Inappropriati RCP Post MR,N. Farmaci Nefrotossici Post MR,N. Farmaci Inappropriati RCP Pre MR,N. Farmaci Nefrotossici Pre MR,N. Farmaci Nefrotossici FUP 1,N. Farmaci Nefrotossici FUP 2,Presente al FUP 1,Presente al FUP 2,Presente in Post MR,Presente in Pre MR
1,id,com_pre,dt_mr,acb_post,n_inapp_beers_post,n_inapp_start_post,n_inapp_stopp_post,n_int_cd_post,acb_pre,n_inapp_beers_pre,...,n_inapp_rcp_post,n_nefro_post,n_inapp_rcp_pre,n_nefro_pre,n_nefro_fu1,n_nefro_fu2,pres_fu1,pres_fu2,pres_post,pres_pre
2,1,Obesità; Ipertensione arteriosa; OSAS; Gozzo m...,2026-05-25 00:00:00; 2026-05-25 00:00:00; 2026...,NaN,NaN,NaN,NaN,Sulfametoxazolo; Trimetroprim; Furosemide; Pan...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,NEFROPATIA ORIGINARIA: nefropatia diabetica in...,2026-05-29 00:00:00; 2026-05-29 00:00:00; 2026...,NaN,NaN,NaN,NaN,Lacidipina; Tacrolimus; Lacidipina; cardioasa,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,Malattia di base: MRC da malattia a depositi d...,2026-05-07 00:00:00; 2026-05-07 00:00:00; 2026...,NaN,NaN,NaN,NaN,CardioASA; Eplerenone; Carvedilolo; Furosemide...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
log_df[log_df["status"] != "mapped"]

,patient_id,patient_sheet_name,new_name,short_name,source,raw_column_found,status
2,1,Foglio1,ACB Score Post MR,acb_post,ACB Score,ACB Score,source_found_but_empty
3,1,Foglio1,N. Farmaci Inappropriati Post MR (Criteri Di ...,n_inapp_beers_post,CRITERI DI BEERS,CRITERI DI BEERS,source_found_but_empty
4,1,Foglio1,N. Farmaci Inappropriati Post MR (Start),n_inapp_start_post,CRITERI START,CRITERI START,source_found_but_empty
5,1,Foglio1,N. Farmaci Inappropriati Post MR (Stopp),n_inapp_stopp_post,CRITERI STOPP,CRITERI STOPP,source_found_but_empty
7,1,Foglio1,ACB Score Pre MR,acb_pre,ACB Score,ACB Score,source_found_but_empty
...,...,...,...,...,...,...,...
337,3,Foglio3,N. Farmaci Nefrotossici FUP 2,n_nefro_fu2,None,None,empty_source_in_LINK
338,3,Foglio3,Presente al FUP 1,pres_fu1,None,None,empty_source_in_LINK
339,3,Foglio3,Presente al FUP 2,pres_fu2,None,None,empty_source_in_LINK
340,3,Foglio3,Presente in Post MR,pres_post,None,None,empty_source_in_LINK


### Examples for conversion rules

In [ ]:
import re
import pandas as pd


def expand_treatments(
    df,
    id_col="id",
    source_col="variable",
    treatment_sep=";",
    field_sep=",",
    output_dosage_unit="g",
    output_time_unit="days"
):
    """
    Convert a column containing one or more treatments into one row per treatment.

    Expected information for each treatment:
        substance, dosage, duration

    Examples:
        "furosemide, 8 mg, 4 days"
        "furosemide 8 mg 4 days"
        "furosemide 8mg 4days + aspirin 100 mg 2 weeks"

    Parameters
    ----------
    df : pandas.DataFrame
        Original dataset.

    id_col : str
        ID column.

    source_col : str
        Column containing the treatment descriptions.

    treatment_sep : str or "no sep"
        Separator between treatments.
        Regardless of this setting, "+" always creates a new row.

    field_sep : str or "no sep"
        Separator between substance, dosage and duration.
        When "no sep", dosage and duration are identified from their units.

    output_dosage_unit : str
        Currently supports "g", "mg", "mcg" or "kg".

    output_time_unit : str
        Currently supports "days", "hours", "weeks" or "years".

    Returns
    -------
    pandas.DataFrame
        One row per treatment, with standardized dosage and duration.
    """

    dosage_to_g = {
        "kg": 1000,
        "g": 1,
        "gram": 1,
        "grams": 1,
        "mg": 1e-3,
        "milligram": 1e-3,
        "milligrams": 1e-3,
        "mcg": 1e-6,
        "µg": 1e-6,
        "ug": 1e-6,
        "microgram": 1e-6,
        "micrograms": 1e-6
    }

    time_to_days = {
        "minute": 1 / 1440,
        "minutes": 1 / 1440,
        "min": 1 / 1440,
        "mins": 1 / 1440,

        "hour": 1 / 24,
        "hours": 1 / 24,
        "hr": 1 / 24,
        "hrs": 1 / 24,
        "h": 1 / 24,

        "day": 1,
        "days": 1,
        "d": 1,

        "week": 7,
        "weeks": 7,
        "wk": 7,
        "wks": 7,

        "month": 30.4375,
        "months": 30.4375,

        "year": 365.25,
        "years": 365.25,
        "yr": 365.25,
        "yrs": 365.25
    }

    dosage_from_g = {
        "kg": 1 / 1000,
        "g": 1,
        "mg": 1000,
        "mcg": 1_000_000,
        "µg": 1_000_000,
        "ug": 1_000_000
    }

    time_from_days = {
        "minutes": 1440,
        "hours": 24,
        "days": 1,
        "weeks": 1 / 7,
        "years": 1 / 365.25
    }

    if output_dosage_unit not in dosage_from_g:
        raise ValueError(
            f"Unsupported output dosage unit: {output_dosage_unit}"
        )

    if output_time_unit not in time_from_days:
        raise ValueError(
            f"Unsupported output time unit: {output_time_unit}"
        )

    dosage_units_pattern = "|".join(
        sorted(map(re.escape, dosage_to_g), key=len, reverse=True)
    )

    time_units_pattern = "|".join(
        sorted(map(re.escape, time_to_days), key=len, reverse=True)
    )

    number_pattern = r"\d+(?:[.,]\d+)?"

    def split_treatments(value):
        if pd.isna(value):
            return []

        value = str(value).strip()

        if not value:
            return []

        # "+" always indicates a new treatment
        separators = [r"\s*\+\s*"]

        if (
            treatment_sep is not None
            and str(treatment_sep).lower() != "no sep"
            and treatment_sep != "+"
        ):
            separators.append(rf"\s*{re.escape(treatment_sep)}\s*")

        split_pattern = "|".join(separators)

        return [
            x.strip()
            for x in re.split(split_pattern, value)
            if x.strip()
        ]

    def parse_with_separator(text):
        parts = [
            x.strip()
            for x in text.split(field_sep, maxsplit=2)
        ]

        if len(parts) != 3:
            return None

        substance, dosage_text, time_text = parts

        dosage_match = re.search(
            rf"(?P<value>{number_pattern})\s*"
            rf"(?P<unit>{dosage_units_pattern})\b",
            dosage_text,
            flags=re.IGNORECASE
        )

        time_match = re.search(
            rf"(?P<value>{number_pattern})\s*"
            rf"(?P<unit>{time_units_pattern})\b",
            time_text,
            flags=re.IGNORECASE
        )

        if not dosage_match or not time_match:
            return None

        return {
            "substance": substance,
            "dosage_value": dosage_match.group("value"),
            "dosage_unit": dosage_match.group("unit"),
            "time_value": time_match.group("value"),
            "time_unit": time_match.group("unit")
        }

    def parse_without_separator(text):
        """
        Find the dosage from its unit, then consider the text after the dosage
        as the location in which the duration must be found.
        """

        dosage_match = re.search(
            rf"(?P<value>{number_pattern})\s*"
            rf"(?P<unit>{dosage_units_pattern})\b",
            text,
            flags=re.IGNORECASE
        )

        if not dosage_match:
            return None

        # Everything before dosage is interpreted as the substance
        substance = text[:dosage_match.start()].strip(" ,;-")

        # Search duration only after the dosage
        remaining_text = text[dosage_match.end():]

        time_match = re.search(
            rf"(?P<value>{number_pattern})\s*"
            rf"(?P<unit>{time_units_pattern})\b",
            remaining_text,
            flags=re.IGNORECASE
        )

        if not time_match:
            return None

        return {
            "substance": substance,
            "dosage_value": dosage_match.group("value"),
            "dosage_unit": dosage_match.group("unit"),
            "time_value": time_match.group("value"),
            "time_unit": time_match.group("unit")
        }

    def parse_treatment(text):
        if (
            field_sep is None
            or str(field_sep).lower() == "no sep"
        ):
            result = parse_without_separator(text)
        else:
            result = parse_with_separator(text)

        if result is None:
            return {
                "substance": pd.NA,
                f"dosage_{output_dosage_unit}": pd.NA,
                f"time_{output_time_unit}": pd.NA,
                "original_value": text,
                "parse_success": False
            }

        dosage_value = float(
            result["dosage_value"].replace(",", ".")
        )
        time_value = float(
            result["time_value"].replace(",", ".")
        )

        dosage_unit = result["dosage_unit"].lower()
        time_unit = result["time_unit"].lower()

        dosage_in_g = dosage_value * dosage_to_g[dosage_unit]
        time_in_days = time_value * time_to_days[time_unit]

        standardized_dosage = (
            dosage_in_g * dosage_from_g[output_dosage_unit]
        )

        standardized_time = (
            time_in_days * time_from_days[output_time_unit]
        )

        return {
            "substance": result["substance"].strip(),
            f"dosage_{output_dosage_unit}": standardized_dosage,
            f"time_{output_time_unit}": standardized_time,
            "original_value": text,
            "parse_success": True
        }

    output_rows = []

    for _, row in df.iterrows():
        treatments = split_treatments(row[source_col])

        for treatment in treatments:
            parsed = parse_treatment(treatment)

            output_row = row.drop(labels=[source_col]).to_dict()
            output_row.update(parsed)
            output_rows.append(output_row)

    result = pd.DataFrame(output_rows)

    dosage_col = f"dosage_{output_dosage_unit}"
    time_col = f"time_{output_time_unit}"

    if not result.empty:
        result[dosage_col] = pd.to_numeric(
            result[dosage_col],
            errors="coerce"
        )

        result[time_col] = pd.to_numeric(
            result[time_col],
            errors="coerce"
        )

    return result

df = pd.DataFrame({
    "id": [1, 2],
    "variable": [
        "furosemide, 8 mg, 4 days + aspirin, 100 mg, 2 weeks",
        "paracetamol, 500 mg, 48 hours"
    ]
})

converted_df = expand_treatments(
    df=df,
    id_col="id",
    source_col="variable",
    treatment_sep=";",
    field_sep=",",
    output_dosage_unit="g",
    output_time_unit="days"
)

print(converted_df)

   id    substance  dosage_g  time_days                 original_value  \
0   1   furosemide     0.008        4.0       furosemide, 8 mg, 4 days   
1   1      aspirin     0.100       14.0       aspirin, 100 mg, 2 weeks   
2   2  paracetamol     0.500        2.0  paracetamol, 500 mg, 48 hours   

   parse_success  
0           True  
1           True  
2           True  


In [ ]:
import re
import pandas as pd
import unicodedata


def extract_drugs_from_text(
    df,
    id_col,
    text_col,
    drug_names,
    keep_duplicates=False,
    keep_original_text=False
):
    """
    Extract drug names from free text and create one row per ID and drug.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    id_col : str
        Name of the ID column.

    text_col : str
        Name of the column containing free text.

    drug_names : list, set or dict
        Drugs to recognize.

        A list or set can contain canonical drug names:

            ["furosemide", "aspirin", "paracetamol"]

        A dictionary can associate alternative names with a canonical name:

            {
                "furosemide": ["furosemide", "lasix"],
                "paracetamol": ["paracetamol", "acetaminophen", "tachipirina"]
            }

    keep_duplicates : bool, default False
        If False, each drug appears only once for each ID.
        If True, repeated mentions are retained.

    keep_original_text : bool, default False
        If True, retain the original text column.

    Returns
    -------
    pandas.DataFrame
        One row per ID and extracted drug.
    """

    if id_col not in df.columns:
        raise ValueError(f"Column '{id_col}' was not found.")

    if text_col not in df.columns:
        raise ValueError(f"Column '{text_col}' was not found.")

    def normalize_text(value):
        """Normalize accents, spaces and letter case."""
        if pd.isna(value):
            return ""

        value = str(value).lower()

        # Convert accented characters to their base form
        value = unicodedata.normalize("NFKD", value)
        value = "".join(
            character
            for character in value
            if not unicodedata.combining(character)
        )

        # Normalize apostrophes and spaces
        value = value.replace("’", "'")
        value = re.sub(r"\s+", " ", value)

        return value.strip()

    # Convert either a list or dictionary into:
    # alternative name -> canonical drug name
    alias_to_canonical = {}

    if isinstance(drug_names, dict):
        for canonical_name, aliases in drug_names.items():

            if isinstance(aliases, str):
                aliases = [aliases]

            # Also recognize the canonical name itself
            all_names = [canonical_name] + list(aliases)

            for name in all_names:
                alias_to_canonical[normalize_text(name)] = canonical_name

    elif isinstance(drug_names, (list, tuple, set)):
        for name in drug_names:
            alias_to_canonical[normalize_text(name)] = name

    else:
        raise TypeError(
            "'drug_names' must be a list, tuple, set or dictionary."
        )

    # Search longer names first.
    # This prevents a shorter name from matching inside a longer name.
    aliases = sorted(
        alias_to_canonical,
        key=len,
        reverse=True
    )

    output_rows = []

    for _, row in df.iterrows():

        original_text = row[text_col]
        normalized_text = normalize_text(original_text)

        if not normalized_text:
            continue

        matches = []

        for alias in aliases:

            # Avoid matching the drug inside another word.
            # It still works with names containing spaces or hyphens.
            pattern = rf"(?<!\w){re.escape(alias)}(?!\w)"

            for match in re.finditer(
                pattern,
                normalized_text,
                flags=re.IGNORECASE
            ):
                matches.append({
                    "drug": alias_to_canonical[alias],
                    "matched_name": match.group(),
                    "position": match.start()
                })

        # Restore the order in which drugs appear in the text
        matches = sorted(matches, key=lambda x: x["position"])

        if not keep_duplicates:
            unique_matches = []
            already_seen = set()

            for match in matches:
                canonical_key = normalize_text(match["drug"])

                if canonical_key not in already_seen:
                    unique_matches.append(match)
                    already_seen.add(canonical_key)

            matches = unique_matches

        for match in matches:
            output_row = {
                id_col: row[id_col],
                "drug": match["drug"]
            }

            if keep_original_text:
                output_row[text_col] = original_text

            output_rows.append(output_row)

    result = pd.DataFrame(output_rows)

    if result.empty:
        columns = [id_col, "drug"]

        if keep_original_text:
            columns.append(text_col)

        return pd.DataFrame(columns=columns)

    return result.reset_index(drop=True)


df = pd.DataFrame({
    "id": [1, 2, 3],
    "clinical_text": [
        "The patient takes furosemide 8 mg and aspirin daily.",
        "Treatment with paracetamol was started. No aspirin was prescribed.",
        "Metformin was suspended and replaced with furosemide."
    ]
})

drug_list = [
    "furosemide",
    "aspirin",
    "paracetamol",
    "metformin"
]

drug_df = extract_drugs_from_text(
    df=df,
    id_col="id",
    text_col="clinical_text",
    drug_names=drug_list
)

print(drug_df)

   id         drug
0   1   furosemide
1   1      aspirin
2   2  paracetamol
3   2      aspirin
4   3    metformin
5   3   furosemide


In [ ]:
df = pd.DataFrame({
    "id": [1001, 1001, 1002, 1003, 1004],
    "clinical_text": [
        (
            "The patient is currently taking Lasix 25 mg once daily for edema "
            "and Cardioaspirin 100 mg once daily for secondary prevention."
        ),
        (
            "At follow-up, furosemide was continued. Metformin was temporarily "
            "suspended because of reduced renal function."
        ),
        (
            "Home therapy includes Tachipirina 1000 mg as needed for pain and "
            "Glucophage 500 mg twice daily."
        ),
        (
            "The patient reports an allergy to aspirin. No acetylsalicylic acid "
            "was prescribed. Paracetamol may be used if necessary."
        ),
        (
            "Acetaminophen was administered in the emergency department. "
            "The patient was not taking Lasix before admission."
        )
    ]
})


drug_dictionary = {
    "furosemide": [
        "furosemide",
        "lasix"
    ],
    "paracetamol": [
        "paracetamol",
        "acetaminophen",
        "tachipirina"
    ],
    "aspirin": [
        "aspirin",
        "acetylsalicylic acid",
        "acido acetilsalicilico",
        "cardioaspirin"
    ],
    "metformin": [
        "metformin",
        "glucophage"
    ]
}

drug_df = extract_drugs_from_text(
    df=df,
    id_col="id",
    text_col="clinical_text",
    drug_names=drug_dictionary,
    keep_duplicates=False,
    keep_original_text=True
)

drug_df_unique = (
    drug_df
    .drop_duplicates(subset=["id", "drug"])
    .reset_index(drop=True)
)

print(drug_df_unique)

     id         drug                                      clinical_text
0  1001   furosemide  The patient is currently taking Lasix 25 mg on...
1  1001      aspirin  The patient is currently taking Lasix 25 mg on...
2  1001    metformin  At follow-up, furosemide was continued. Metfor...
3  1002  paracetamol  Home therapy includes Tachipirina 1000 mg as n...
4  1002    metformin  Home therapy includes Tachipirina 1000 mg as n...
5  1003      aspirin  The patient reports an allergy to aspirin. No ...
6  1003  paracetamol  The patient reports an allergy to aspirin. No ...
7  1004  paracetamol  Acetaminophen was administered in the emergenc...
8  1004   furosemide  Acetaminophen was administered in the emergenc...


In [ ]:
df = pd.DataFrame({
    "id": [1, 1, 2],
    "clinical_text": [
        "The patient takes aspirin.",
        "Furosemide was added. Aspirin was continued.",
        "Paracetamol as needed."
    ]
})

drug_df = extract_drugs_from_text(
    df=df,
    id_col="id",
    text_col="clinical_text",
    drug_names=drug_dictionary
)

# Ensure only one row per ID and drug across the whole dataset
drug_df = (
    drug_df
    .drop_duplicates(subset=["id", "drug"])
    .reset_index(drop=True)
)

print(drug_df)

   id         drug
0   1      aspirin
1   1   furosemide
2   2  paracetamol


### One function conversion

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from openpyxl import load_workbook

SUPPORTED_EXTENSIONS = {'.xls', '.xlsx', '.ods'}
NEGATION_PHRASES = (
    'no', 'non', 'nessun', 'nessuna', 'nega', 'assenza di', 'senza',
    'senza evidenza di', 'escluso', 'negative for',
)
FAMILY_PHRASES = (
    'familiarita per', 'anamnesi familiare', 'madre con', 'padre con',
    'family history of',
)
UNCERTAIN_PHRASES = (
    'sospetto', 'possibile', 'probabile', 'da escludere', 'rule out', 'query',
)
COMORBIDITY_DOMINANCE = {
    'FIBRILLAZIONE ATRIALE': {'ARITMIA'},
    'INSUFFICIENZA CARDIACA': {'SCOMPENSO CARDIACO'},
    'DISTURBI DEL MOVIMENTO': {'DISTURBO NEUROLOGICO'},
    'DEMENZA/ALZHEIMER': {'DISTURBO NEUROLOGICO'},
    'DEPRESSIONE': {'DISTURBI DELL\'UMORE'},
}
INTERACTION_PRIORITY = {
    'EMORRAGIA E SANGUINAMENTO': 10,
    'PROLUNGAMENTO QT': 10,
    'IPOTENSIONE': 10,
    'SINCOPE': 10,
    'DEPRESSIONE SNC': 10,
    'MIELOTOSSICITÀ/CITOPENIA': 10,
    'NEFROTOSSICITÀ': 10,
    'EPATOTOSSICITÀ': 10,
    'MIOPATIA/DANNO MUSCOLARE': 10,
    'ALTERAZIONE ELETTROLITICA': 10,
    'IPERGLICEMIA': 10,
    'IPOGLICEMIA': 10,
    'AUMENTO TRANSAMINASI': 20,
    'AUMENTO RISCHIO DI SANGUINAMENTO': 30,
    'AUMENTO RISCHIO CADUTE': 30,
    'RISCHIO FRATTURE': 30,
    'DIMINUZIONE EFFICACIA': 40,
    'ALTERAZIONE ASSORBIMENTO': 45,
    'AUMENTO DELL\'ESPOSIZIONE O DELL\'EFFETTO DI UNO DEI FARMACI': 50,
    'AUMENTO DELLA TOSSICITÀ DI UNO DEI FARMACI': 90,
    'AUMENTO RISCHIO CV': 90,
    'AUMENTO RISCHIO DI EVENTI AVVERSI GASTROINTESTINALI': 90,
    'ALTRO ESITO CLINICAMENTE RILEVANTE': 100,
}


def get_engine(path: Path | str) -> str:
    suffix = Path(path).suffix.lower()
    engines = {'.xlsx': 'openpyxl', '.xls': 'xlrd', '.ods': 'odf'}
    if suffix not in engines:
        raise ValueError(f'Unsupported workbook extension: {suffix}')
    return engines[suffix]

def clean_string(value):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    return value or None

def normalize_text(value) -> str:
    value = clean_string(value)
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', value.lower())
    value = ''.join(c for c in value if not unicodedata.combining(c))
    value = value.replace('’', "'")
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def normalize_name(value) -> str:
    return normalize_text(value).upper()

def split_terms(value, separators=r'[;,]') -> list[str]:
    text = clean_string(value)
    if not text:
        return []
    return [normalize_text(x) for x in re.split(separators, text) if normalize_text(x)]

def boundary_contains(text: str, term: str) -> bool:
    return bool(term and re.search(rf'(?<!\w){re.escape(term)}(?!\w)', text))

def make_unique_columns(columns: Iterable) -> list[str]:
    seen = {}
    out = []
    for value in columns:
        name = clean_string(value) or 'Unnamed'
        seen[name] = seen.get(name, 0) + 1
        out.append(name if seen[name] == 1 else f'{name}__{seen[name]}')
    return out

def read_excel(path: Path | str, **kwargs):
    return pd.read_excel(path, engine=get_engine(path), **kwargs)

def preceding_context(text: str, start: int, max_words: int = 7) -> str:
    prefix = text[:start].strip().split()
    return ' '.join(prefix[-max_words:])

def match_is_blocked(text: str, term: str) -> bool:
    """Exclude negated, family-history, or uncertain mentions."""
    for match in re.finditer(rf'(?<!\w){re.escape(term)}(?!\w)', text):
        context = preceding_context(text, match.start(), max_words=7)
        if any(boundary_contains(context, phrase) for phrase in NEGATION_PHRASES):
            continue
        if any(phrase in context for phrase in FAMILY_PHRASES):
            continue
        if any(boundary_contains(context, phrase) for phrase in UNCERTAIN_PHRASES):
            continue
        return False
    return True

def safe_terms(value, separators=r';') -> set[str]:
    return {x for x in split_terms(value, separators=separators) if len(x) >= 2}


@dataclass(frozen=True)
class DrugInfo:
    substance: str
    atc: str | None


@dataclass(frozen=True)
class DictionaryEntry:
    canonical: str
    automatic_terms: tuple[str, ...]
    roots: tuple[str, ...]
    blocked_terms: tuple[str, ...]
    order: int

class DictionaryCache:
    """Load the current dictionaries once and build indexes for matching."""

    def __init__(self, drugs_path: Path, com_path: Path, interact_path: Path):
        self.drug_aliases: dict[str, DrugInfo] = {}
        self.drug_candidates: dict[str, list[str]] = {}
        self.com_entries: list[DictionaryEntry] = []
        self.interaction_entries: list[DictionaryEntry] = []
        self._load_drugs(drugs_path)
        self._load_comorbidities(com_path)
        self._load_interactions(interact_path)

    def _load_drugs(self, path: Path):
        df = read_excel(path, sheet_name=0).fillna('')
        required = {'canonical_substance', 'aliases'}
        if not required.issubset(df.columns):
            raise ValueError(f'Drugs_dict.xlsx must contain {sorted(required)}')

        canonical_terms = []
        alias_terms = []
        for _, row in df.iterrows():
            canonical = clean_string(row['canonical_substance'])
            if not canonical:
                continue
            info = DrugInfo(canonical, clean_string(row.get('atc_code')))
            canonical_term = normalize_text(canonical)
            if canonical_term:
                canonical_terms.append((canonical_term, info))
            terms = safe_terms(row.get('aliases')) | safe_terms(row.get('match_roots'))
            alias_terms.extend((term, info) for term in terms)

        # Exact canonical names override aliases from combination products.
        for term, info in alias_terms:
            self.drug_aliases.setdefault(term, info)
        for term, info in canonical_terms:
            self.drug_aliases[term] = info

        for alias in self.drug_aliases:
            for token in set(alias.split()):
                if len(token) >= 3:
                    self.drug_candidates.setdefault(token, []).append(alias)
        for token in self.drug_candidates:
            self.drug_candidates[token].sort(key=len, reverse=True)

    @staticmethod
    def _automatic_terms(row, aliases_col, roots_col, usage_col,
                         review_col, non_auto_col) -> tuple[tuple[str, ...], tuple[str, ...], tuple[str, ...]]:
        usage = normalize_name(row.get(usage_col, ''))
        aliases = safe_terms(row.get(aliases_col))
        roots = safe_terms(row.get(roots_col))
        review = safe_terms(row.get(review_col))
        non_auto = safe_terms(row.get(non_auto_col))
        blocked = review | non_auto

        # NO and REVISIONE-only rows must never be assigned automatically.
        automatic_allowed = usage.startswith('SI')
        if not automatic_allowed:
            return (), (), tuple(sorted(aliases | roots | blocked, key=len, reverse=True))

        aliases -= blocked
        roots -= blocked
        return (
            tuple(sorted(aliases, key=len, reverse=True)),
            tuple(sorted(roots, key=len, reverse=True)),
            tuple(sorted(blocked, key=len, reverse=True)),
        )

    def _load_comorbidities(self, path: Path):
        df = read_excel(path, sheet_name=0).fillna('')
        required = {'canonical_name', 'aliases', 'match_roots'}
        if not required.issubset(df.columns):
            raise ValueError(f'Com_dict.xlsx must contain {sorted(required)}')

        for order, (_, row) in enumerate(df.iterrows()):
            canonical = clean_string(row['canonical_name'])
            if not canonical:
                continue
            aliases, roots, blocked = self._automatic_terms(
                row, 'aliases', 'match_roots', 'uso_automatico',
                'aliases_da_revisionare', 'aliases_non_automatici',
            )
            self.com_entries.append(DictionaryEntry(canonical, aliases, roots, blocked, order))

    def _load_interactions(self, path: Path):
        df = read_excel(path, sheet_name=0).fillna('')
        if not {'Conseguenze', 'Effetti'}.issubset(df.columns):
            raise ValueError("Interact_dict.xlsx must contain 'Conseguenze' and 'Effetti'")

        for order, (_, row) in enumerate(df.iterrows()):
            canonical = clean_string(row['Conseguenze'])
            if not canonical:
                continue
            usage = normalize_name(row.get('Uso automatico', ''))
            effects_set = safe_terms(row.get('Effetti'), separators=r',')
            review = safe_terms(row.get('Termini da revisionare'), separators=r',')
            non_auto = safe_terms(row.get('Termini non automatici'), separators=r',')
            blocked_set = review | non_auto
            effects_set -= blocked_set
            effects = tuple(sorted(effects_set, key=len, reverse=True)) if usage.startswith('SI') else ()
            blocked = tuple(sorted(blocked_set, key=len, reverse=True))
            self.interaction_entries.append(DictionaryEntry(canonical, effects, (), blocked, order))

    def lookup_drug(self, text) -> DrugInfo | None:
        normalized = normalize_text(text)
        if not normalized:
            return None
        if normalized in self.drug_aliases:
            return self.drug_aliases[normalized]

        candidates = set()
        for token in normalized.split():
            candidates.update(self.drug_candidates.get(token, ()))
        for alias in sorted(candidates, key=len, reverse=True):
            if boundary_contains(normalized, alias):
                return self.drug_aliases[alias]
        return None

    def find_drugs(self, text) -> list[DrugInfo]:
        normalized = normalize_text(text)
        if not normalized:
            return []
        candidates = set()
        for token in normalized.split():
            candidates.update(self.drug_candidates.get(token, ()))
        found = {}
        for alias in sorted(candidates, key=len, reverse=True):
            if boundary_contains(normalized, alias):
                info = self.drug_aliases[alias]
                found.setdefault(normalize_text(info.substance), info)
        return list(found.values())

    @staticmethod
    def _entry_matches(normalized: str, entry: DictionaryEntry) -> bool:
        for term in entry.automatic_terms:
            if boundary_contains(normalized, term) and not match_is_blocked(normalized, term):
                return True
        for root in entry.roots:
            # Roots can be stems, therefore boundary matching is not required.
            for match in re.finditer(re.escape(root), normalized):
                context = preceding_context(normalized, match.start(), max_words=7)
                if any(boundary_contains(context, x) for x in NEGATION_PHRASES):
                    continue
                if any(x in context for x in FAMILY_PHRASES):
                    continue
                if any(boundary_contains(context, x) for x in UNCERTAIN_PHRASES):
                    continue
                return True
        return False

    def find_comorbidities(self, text) -> list[str]:
        normalized = normalize_text(text)
        if not normalized:
            return []

        found = [entry.canonical for entry in self.com_entries if self._entry_matches(normalized, entry)]

        # The current reference groups all infection sites into one category.
        infection_matches = [x for x in found if x.startswith('INFEZIONE') or 'MALATTIA INFETTIVA' in x]
        if infection_matches:
            preferred = next((x for x in found if x == 'INFEZIONE/MALATTIA INFETTIVA'), infection_matches[0])
            found = [x for x in found if x not in infection_matches] + [preferred]

        # Apply specific-over-generic collision rules.
        found_set = set(found)
        for specific, generic_set in COMORBIDITY_DOMINANCE.items():
            if specific in found_set:
                found_set -= generic_set

        # Residual ALTRO is never automatic.
        found_set.discard('ALTRO')
        output = []
        for value in found:
            if value in found_set and value not in output:
                output.append(value)
        return output

    def has_comorbidity(self, text, target: str) -> bool:
        return normalize_text(target) in {normalize_text(x) for x in self.find_comorbidities(text)}

    def find_consequences(self, text, max_items=3) -> list[str]:
        normalized = normalize_text(text)
        if not normalized:
            return []

        matches = [entry for entry in self.interaction_entries if self._entry_matches(normalized, entry)]
        names = {x.canonical for x in matches}

        # Small grammatical heuristics cover common Italian inflections that
        # cannot be represented reliably by exact comma-separated aliases.
        if re.search(r'\b(ast|alt|got|gpt)\b.{0,25}\b(aumentat|elevat)', normalized) or re.search(
            r'\b(aumentat|elevat)\b.{0,25}\b(ast|alt|got|gpt)\b', normalized
        ):
            names.add('AUMENTO TRANSAMINASI')
        if re.search(r'\b(aumenta|aumentano|incrementa|incrementano)\b.{0,35}\b(concentrazion|livell)', normalized):
            names.add('AUMENTO DELL\'ESPOSIZIONE O DELL\'EFFETTO DI UNO DEI FARMACI')

        risk_bleeding = bool(re.search(r'\brischio\b.{0,30}\b(sanguin|emorrag)', normalized))
        manifest_bleeding = bool(re.search(
            r'\b(emorragia|sanguinamento|melena|ematemesi|ematuria|epistassi|ematoma|ecchimosi)\b',
            normalized,
        )) and not risk_bleeding
        if risk_bleeding:
            names.add('AUMENTO RISCHIO DI SANGUINAMENTO')
            names.discard('EMORRAGIA E SANGUINAMENTO')
        elif manifest_bleeding and 'EMORRAGIA E SANGUINAMENTO' in names:
            names.discard('AUMENTO RISCHIO DI SANGUINAMENTO')

        # Organ damage prevails over laboratory-only change.
        if 'EPATOTOSSICITÀ' in names:
            names.discard('AUMENTO TRANSAMINASI')
        # Specific toxicity prevails over generic exposure/toxicity categories.
        specific = {x for x in names if INTERACTION_PRIORITY.get(x, 50) <= 30}
        if specific:
            names.discard('AUMENTO DELL\'ESPOSIZIONE O DELL\'EFFETTO DI UNO DEI FARMACI')
            names.discard('AUMENTO DELLA TOSSICITÀ DI UNO DEI FARMACI')

        ordered = sorted(
            names,
            key=lambda name: (
                INTERACTION_PRIORITY.get(name, 50),
                next((e.order for e in matches if e.canonical == name), 9999),
            ),
        )
        return ordered[:max_items]

class RuleResult:
    def __init__(self, data=None, status='mapped', message=None):
        self.data = pd.DataFrame() if data is None else data.reset_index(drop=True)
        self.status = status
        self.message = message

class ConverterEngine:
    # The look-ahead accepts compact strings such as '5 mg1 cp' while avoiding
    # partial unit matches inside alphabetic words.
    DOSAGE_PATTERN = re.compile(
        r'(?P<value>\d+(?:[.,]\d+)?)\s*(?P<unit>kg|mg|mcg|ug|µg|g|ui|iu)(?=$|[^a-zA-Zµ])',
        re.I,
    )
    ROUTE_PATTERN = re.compile(r'\b(per\s+os|orale|os|ev|iv|im|sc|sottocute|sottocutanea|inalatoria|topica|transdermica)\b', re.I)

    def __init__(self, dictionaries: DictionaryCache):
        self.d = dictionaries

    def apply(self, rule, values, parameters):
        method = getattr(self, f'rule_{rule}', None)
        if method is None:
            raise ValueError(f'Unknown rule: {rule}')
        return method(values, parameters)

    @staticmethod
    def cleaned(values):
        return [str(v).strip() for v in values if v is not None and not pd.isna(v) and str(v).strip()]

    def rule_parse_numeric(self, values, p):
        vals = self.cleaned(values)
        if not vals:
            return RuleResult(pd.DataFrame({'value': [np.nan]}), 'source_found_but_empty')
        raw = vals[0].replace(',', '.')
        m = re.search(r'-?\d+(?:\.\d+)?', raw)
        value = pd.to_numeric(m.group() if m else raw, errors='coerce')
        return RuleResult(pd.DataFrame({'value': [value]}), 'mapped' if pd.notna(value) else 'numeric_parse_failed')

    def rule_parse_date(self, values, p):
        vals = self.cleaned(values)
        value = (
            pd.to_datetime(
                vals[0],
                dayfirst=p.get('dayfirst', True),
                errors=p.get('errors', 'coerce'),
            )
            if vals
            else pd.NaT
        )
        if pd.isna(value):
            return RuleResult(
                pd.DataFrame({'value': [np.nan]}),
                'date_parse_failed',
            )

        # Dates are intentionally exported as ISO strings rather than Excel
        # datetimes, preventing values such as '2026-05-25 00:00:00'.
        date_format = p.get('format', '%Y-%m-%d')
        return RuleResult(pd.DataFrame({'value': [value.strftime(date_format)]}))

    def rule_calculate_age(self, values, p):
        vals = self.cleaned(values)
        birth = pd.to_datetime(vals[0], dayfirst=p.get('dayfirst', True), errors='coerce') if vals else pd.NaT
        ref = pd.to_datetime(p.get('reference_date', '2026-01-01'), errors='coerce')
        if pd.isna(birth) or pd.isna(ref):
            return RuleResult(pd.DataFrame({'age_years': [np.nan]}), 'age_calculation_failed')
        age = ref.year - birth.year - int((ref.month, ref.day) < (birth.month, birth.day))
        return RuleResult(pd.DataFrame({'age_years': [age]}))

    def rule_count_nonempty_rows(self, values, p):
        return RuleResult(pd.DataFrame({'count': [len(self.cleaned(values))]}))

    def rule_count_drug_mentions(self, values, p):
        unique = {
            normalize_text(info.substance)
            for value in self.cleaned(values)
            for info in self.d.find_drugs(value)
        }
        return RuleResult(pd.DataFrame({'count': [len(unique)]}))

    def rule_count_comorbidities(self, values, p):
        unique = {
            comorbidity
            for value in self.cleaned(values)
            for comorbidity in self.d.find_comorbidities(value)
            if comorbidity not in {'FUMO', 'ALCOL', 'ALCOOL'}
        }
        return RuleResult(pd.DataFrame({'count': [len(unique)]}))

    def rule_row_aligned_value(self, values, p):
        rows = [{'value': clean_string(value), '_source_row': idx} for idx, value in enumerate(values)]
        df = pd.DataFrame(rows)
        if p.get('forward_fill') and not df.empty:
            df['value'] = df['value'].ffill()
        if p.get('drop_empty', True) and not df.empty:
            df = df[df['value'].notna()]
        return RuleResult(df, 'mapped' if not df.empty else 'source_found_but_empty')

    def rule_comorbidity_dictionary_lookup(self, values, p):
        rows = []
        for idx, value in enumerate(values):
            for subidx, comorbidity in enumerate(self.d.find_comorbidities(value)):
                if comorbidity in {'FUMO', 'ALCOL', 'ALCOOL'}:
                    continue
                rows.append({'comorbidity': comorbidity, '_source_row': idx, '_source_subrow': subidx})
        df = pd.DataFrame(rows)
        if not p.get('keep_duplicates', False) and not df.empty:
            df = df.drop_duplicates('comorbidity', keep='first')
        return RuleResult(df, 'mapped' if not df.empty else 'no_comorbidities_extracted')

    def rule_comorbidity_flag_lookup(self, values, p):
        target = p['target']
        found = any(self.d.has_comorbidity(value, target) for value in self.cleaned(values))
        return RuleResult(pd.DataFrame({'value': [1 if found else 0]}))

    def rule_interaction_dictionary_lookup(self, values, p):
        rows = []
        max_items = int(p.get('max_consequences', 3))
        for idx, value in enumerate(values):
            consequences = self.d.find_consequences(value, max_items=max_items)
            row = {'_source_row': idx}
            for j in range(1, max_items + 1):
                row[f'consequence_{j}'] = consequences[j - 1] if j <= len(consequences) else np.nan
            rows.append(row)
        df = pd.DataFrame(rows)
        return RuleResult(df, 'mapped' if not df.empty else 'source_found_but_empty')

    def rule_drug_dictionary_lookup(self, values, p):
        rows = []
        for idx, value in enumerate(values):
            info = self.d.lookup_drug(value)
            rows.append({
                'substance': info.substance if info else np.nan,
                'atc': info.atc if info else np.nan,
                '_source_row': idx,
            })
        df = pd.DataFrame(rows)
        status = 'mapped' if not df.empty and df['substance'].notna().any() else 'dictionary_lookup_failed'
        return RuleResult(df, status)

    def rule_parse_treatment_row(self, values, p):
        rows = []
        for idx, value in enumerate(values):
            text = clean_string(value)
            if not text:
                continue
            dose = self.DOSAGE_PATTERN.search(text)
            route = self.ROUTE_PATTERN.search(text)
            dosage_value = float(dose.group('value').replace(',', '.')) if dose else np.nan
            dosage_unit = dose.group('unit').lower() if dose else np.nan
            daily_dose = self._daily_dose(text, dosage_value)
            rows.append({
                'dosage_value': dosage_value,
                'dosage_unit': dosage_unit,
                'daily_dose': daily_dose,
                'route': route.group().strip() if route else np.nan,
                '_source_row': idx,
            })
        df = pd.DataFrame(rows)
        return RuleResult(df, 'mapped' if not df.empty else 'source_found_but_empty')

    @staticmethod
    def _daily_dose(text, dose):
        """Return administrations per day, not milligrams per day.

        Examples:
        - '5 mg 1 cp ore 8' -> 1
        - '180 mg ore 8 e ore 20' -> 2
        - '2,5 mg (2 cp da 1 mg) ore 8' -> 1
        - '5 mg/24ore cerotto transdermico' -> 1
        """
        if pd.isna(dose):
            return np.nan

        raw = str(text).lower().replace('’', "'")
        normalized = normalize_text(text)

        # Explicit frequency always has priority.
        explicit_patterns = [
            r'(\d+(?:[.,]\d+)?)\s*volte\s*(?:al|a|il)?\s*(?:giorno|die)',
            r'(\d+(?:[.,]\d+)?)\s*(?:x|per)\s*(?:die|giorno)',
        ]
        for pattern in explicit_patterns:
            match = re.search(pattern, raw, flags=re.I)
            if match:
                return float(match.group(1).replace(',', '.'))

        every_hours = re.search(r'ogni\s*(\d+(?:[.,]\d+)?)\s*ore?', raw, flags=re.I)
        if every_hours:
            hours = float(every_hours.group(1).replace(',', '.'))
            return round(24 / hours, 6) if hours > 0 else np.nan

        # A formulation lasting 24 hours represents one daily administration.
        if re.search(r'(?:/|ogni\s*)24\s*ore?', raw, flags=re.I):
            return 1

        # Count distinct administration times. Repeated mentions of the same
        # hour do not inflate the frequency.
        hours = re.findall(r'\bore?\s*(\d{1,2})(?::([0-5]\d))?', raw, flags=re.I)
        if hours:
            unique_hours = {(int(hour), minute or '00') for hour, minute in hours}
            return len(unique_hours)

        # Common textual schedules.
        if any(phrase in normalized for phrase in [
            'tre volte al giorno', 'ter die', 'tid',
            'mattina pomeriggio sera',
        ]):
            return 3
        if any(phrase in normalized for phrase in [
            'due volte al giorno', 'bis die', 'bid',
            'mattina e sera', 'mattino e sera',
        ]):
            return 2
        if any(phrase in normalized for phrase in [
            'una volta al giorno', '1 die', 'qd',
            'al mattino', 'alla sera', 'la sera', 'la mattina',
        ]):
            return 1

        # A single tablet/capsule/patch instruction without another frequency
        # indicator is interpreted as one administration per day. Tablet count
        # is deliberately not used as frequency: '2 cp da 1 mg ore 8' is one dose.
        if re.search(r'\b\d+(?:[.,]\d+)?\s*(?:cp|cpr|compress[ae]|capsul[ae]|cerott[oi])\b', raw, flags=re.I):
            return 1

        return np.nan

def parse_parameters(value):
    if value is None or pd.isna(value) or str(value).strip() == '':
        return {}
    return json.loads(value) if not isinstance(value, dict) else dict(value)

def read_link(link_path):
    link = read_excel(link_path, sheet_name='LINK').fillna('')
    required = [
        'sheet_name', 'group_name', 'old_name', 'new_name', 'short_name',
        'source', 'rule', 'rule_group', 'output_field', 'expansion_group',
        'rule_parameters',
    ]
    missing = [x for x in required if x not in link.columns]
    if missing:
        raise ValueError(f'Missing LINK columns: {missing}')
    link = link[(link['short_name'] != '') & (link['rule'] != '')].copy().reset_index(drop=True)
    link['_link_id'] = [f'__link_{i:04d}' for i in range(len(link))]
    return link

def read_patient_sheets(path):
    book = pd.ExcelFile(path, engine=get_engine(path))
    out = {}
    for sheet in book.sheet_names:
        # Patient workbooks use row 2 as the variable-name row.
        df = read_excel(path, sheet_name=sheet, header=1)
        df.columns = make_unique_columns(df.columns)
        out[sheet] = df
    return out

def find_source(df, source):
    lookup = {normalize_name(c.split('__')[0]): c for c in df.columns}
    col = lookup.get(normalize_name(source))
    return (col, df[col]) if col is not None else (None, None)

def combine_frames(frames):
    if not frames:
        return pd.DataFrame(index=[0])
    keyed, singles = [], []
    for frame in frames:
        frame = frame.copy()
        keys = [x for x in ['_source_row', '_source_subrow'] if x in frame.columns]
        if keys:
            keyed.append(frame.drop_duplicates(keys).set_index(keys))
        elif len(frame) == 1:
            singles.append(frame.reset_index(drop=True))
        else:
            keyed.append(frame)
    result = pd.concat(keyed, axis=1, join='outer').reset_index() if keyed else pd.DataFrame(index=[0])
    result = result.loc[:, ~result.columns.duplicated()]
    n = max(len(result), 1)
    for frame in singles:
        repeated = pd.concat([frame] * n, ignore_index=True)
        result = pd.concat([result.reset_index(drop=True), repeated], axis=1)
    return result.loc[:, ~result.columns.duplicated()]

def execute_patient(patient_id, sheet_name, df, link, engine):
    """Execute LINK rules and return one frame for each expansion group."""
    logs = []
    expansion_frames = {}

    for expansion, expansion_df in link.groupby('expansion_group', sort=False):
        group_outputs = []
        for _, group in expansion_df.groupby('rule_group', sort=False):
            execution_outputs = []
            for (source, rule, params_raw), rows in group.groupby(
                ['source', 'rule', 'rule_parameters'], sort=False, dropna=False
            ):
                source_col, values = find_source(df, source)
                if source_col is None:
                    result = RuleResult(pd.DataFrame(index=[0]), 'source_not_found_in_patient_sheet')
                else:
                    result = engine.apply(rule, values, parse_parameters(params_raw))

                out = pd.DataFrame(index=result.data.index if not result.data.empty else [0])
                for key in ['_source_row', '_source_subrow']:
                    if key in result.data.columns:
                        out[key] = result.data[key].values

                for _, link_row in rows.iterrows():
                    field = link_row['output_field']
                    link_id = link_row['_link_id']
                    out[link_id] = result.data[field].values if field in result.data.columns else np.nan
                    logs.append({
                        'patient_id': patient_id,
                        'patient_sheet_name': sheet_name,
                        'target_sheet': link_row['sheet_name'],
                        'group_name': link_row['group_name'],
                        'old_name': link_row['old_name'],
                        'new_name': link_row['new_name'],
                        'short_name': link_row['short_name'],
                        'source': source,
                        'raw_column_found': source_col,
                        'rule': rule,
                        'status': result.status,
                        'n_extracted_rows': len(result.data),
                        'message': result.message,
                    })
                execution_outputs.append(out)
            group_outputs.append(combine_frames(execution_outputs))
        expansion_frames[expansion] = combine_frames(group_outputs)

    expansion_frames.setdefault('patient', pd.DataFrame(index=[0]))
    expansion_frames['patient'] = expansion_frames['patient'].iloc[[0]].reset_index(drop=True)
    return expansion_frames, logs


@dataclass
class TemplateSheet:
    name: str
    header_rows: int
    groups: list[str]
    old_headers: list[str]
    output_headers: list[str]
    link_ids: list[str | None]

class OutputTemplate:
    """Read output structure from dataset_finale_base.xlsx."""

    def __init__(self, template_path: Path, link: pd.DataFrame):
        self.path = Path(template_path)
        self.sheet_order: list[str] = []
        self.sheets: dict[str, TemplateSheet] = {}
        self._load(link)

    @staticmethod
    def _find_link_row(link, sheet_name, group_name, old_name):
        sheet_candidates = link[
            link['sheet_name'].map(normalize_name) == normalize_name(sheet_name)
        ].copy()
        if group_name:
            # In GENERALE, the same old label occurs at several timepoints.
            # A mapping is valid only for the explicitly named group.
            sheet_candidates = sheet_candidates[
                sheet_candidates['group_name'].map(normalize_name) == normalize_name(group_name)
            ]

        old_norm = normalize_name(old_name)
        exact = sheet_candidates[
            sheet_candidates['old_name'].map(normalize_name) == old_norm
        ]
        if not exact.empty:
            return exact.iloc[0]

        # Permit explanatory suffixes in the template, e.g.
        # 'COMORBIDITA PRE MR (MH: dizionario da fare)'.
        prefixed = sheet_candidates[
            sheet_candidates['old_name'].map(
                lambda x: bool(normalize_name(x)) and old_norm.startswith(normalize_name(x))
            )
        ]
        return None if prefixed.empty else prefixed.iloc[0]

    def _load(self, link):
        wb = load_workbook(self.path, read_only=True, data_only=False)
        for ws in wb.worksheets:
            self.sheet_order.append(ws.title)
            header_rows = 2 if normalize_name(ws.title) == 'GENERALE' else 1
            max_col = ws.max_column
            groups = []
            old_headers = []
            output_headers = []
            link_ids = []

            for col in range(1, max_col + 1):
                group = clean_string(ws.cell(1, col).value) if header_rows == 2 else None
                old_header = clean_string(ws.cell(header_rows, col).value)
                groups.append(group or '')
                old_headers.append(old_header or '')

                matched = self._find_link_row(link, ws.title, group or '', old_header or '')
                if matched is None:
                    output_headers.append(old_header or '')
                    link_ids.append(None)
                else:
                    output_headers.append(clean_string(matched['new_name']) or old_header or '')
                    link_ids.append(matched['_link_id'])

            self.sheets[ws.title] = TemplateSheet(
                name=ws.title,
                header_rows=header_rows,
                groups=groups,
                old_headers=old_headers,
                output_headers=output_headers,
                link_ids=link_ids,
            )
        wb.close()

def _sheet_expansion_groups(sheet_name, link):
    rows = link[link['sheet_name'].map(normalize_name) == normalize_name(sheet_name)]
    groups = [x for x in rows['expansion_group'].tolist() if clean_string(x)]
    return list(dict.fromkeys(groups))

def make_patient_sheet_frame(patient_id, template_sheet, link, expansion_frames):
    relevant_groups = _sheet_expansion_groups(template_sheet.name, link)
    non_patient_groups = [x for x in relevant_groups if x != 'patient']

    if non_patient_groups:
        base = combine_frames([expansion_frames.get(x, pd.DataFrame(index=[0])) for x in non_patient_groups])
    else:
        base = pd.DataFrame(index=[0])

    if base.empty:
        base = pd.DataFrame(index=[0])
    base = base.reset_index(drop=True)
    n = max(len(base), 1)

    # Broadcast patient-level values onto expanded sheets (e.g. DATA RICOGNIZIONE).
    patient_frame = expansion_frames.get('patient', pd.DataFrame(index=[0])).iloc[[0]].reset_index(drop=True)
    if not patient_frame.empty:
        patient_repeated = pd.concat([patient_frame] * n, ignore_index=True)
        base = pd.concat([base, patient_repeated], axis=1)
        base = base.loc[:, ~base.columns.duplicated()]

    output = pd.DataFrame(index=range(n))
    for idx, (old_header, link_id) in enumerate(zip(template_sheet.old_headers, template_sheet.link_ids)):
        internal_col = f'__template_col_{idx:03d}'
        if normalize_name(old_header) == 'ID PAZIENTE':
            output[internal_col] = [patient_id] * n
        elif link_id and link_id in base.columns:
            output[internal_col] = base[link_id].reindex(range(n)).values
        else:
            output[internal_col] = [np.nan] * n

    if normalize_name(template_sheet.name) == 'INTERAZIONI':
        output = clean_interaction_rows(output, template_sheet)

    return output.reset_index(drop=True)

def clean_interaction_rows(output, template_sheet):
    """Remove artifacts created by forward-filled interaction drug columns.

    Drug A and Drug B are deliberately forward-filled because merged cells are
    common in the source workbook.  This can create trailing rows containing
    only the same drug pair while type, reason, consequence, and clinical advice
    are all empty.  Such rows are removed when a populated record for the same
    pair exists.  A pair-only row is retained when it is the sole record for
    that pair, so potentially useful source information is not discarded.
    """
    if output.empty:
        return output

    def matching_columns(*terms):
        cols = []
        for idx, (old_header, new_header) in enumerate(
            zip(template_sheet.old_headers, template_sheet.output_headers)
        ):
            label = f"{old_header} {new_header}"
            normalized = normalize_name(label)
            if any(normalize_name(term) in normalized for term in terms):
                cols.append(f'__template_col_{idx:03d}')
        return cols

    drug_1_cols = matching_columns('FARMACO INTERAGENTE 1', 'FARMACO INTERAGENTE_1')
    drug_2_cols = matching_columns('FARMACO INTERAGENTE 2', 'FARMACO INTERAGENTE_2')
    pair_cols = (drug_1_cols[:1] + drug_2_cols[:1])

    detail_cols = matching_columns(
        'TIPO INTERAZIONE',
        'TIPO DI INTERAZIONE',
        'MOTIVO INTERAZIONE',
        'MOTIVO DELL INTERAZIONE',
        'CONSEGUENZA INTERAZIONE',
        'CONSEGUENZA DELL INTERAZIONE',
        'CONSIGLIO CLINICO',
    )

    if len(pair_cols) < 2:
        return output.drop_duplicates().reset_index(drop=True)

    def populated(value):
        return clean_string(value) is not None

    pair_keys = output[pair_cols].apply(
        lambda row: tuple(normalize_text(value) for value in row), axis=1
    )
    has_pair = output[pair_cols].apply(lambda row: any(populated(v) for v in row), axis=1)
    has_details = (
        output[detail_cols].apply(lambda row: any(populated(v) for v in row), axis=1)
        if detail_cols else pd.Series(False, index=output.index)
    )

    populated_pairs = set(pair_keys[has_pair & has_details])
    inherited_blank = has_pair & ~has_details & pair_keys.isin(populated_pairs)
    cleaned = output.loc[~inherited_blank].copy()

    # Remove fully identical interaction records, while retaining the first
    # occurrence and the original clinical order.
    cleaned = cleaned.drop_duplicates(keep='first')
    return cleaned.reset_index(drop=True)

def excel_value(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, np.generic):
        return value.item()
    return value

def write_template_output(template_path, output_path, template, sheet_frames):
    """Copy the template, retain its formatting, rename mapped headers, and add data."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(template_path, output_path)
    wb = load_workbook(output_path)

    for sheet_name in template.sheet_order:
        ws = wb[sheet_name]
        spec = template.sheets[sheet_name]

        # Remove any old data while preserving the template header/style rows.
        if ws.max_row > spec.header_rows:
            ws.delete_rows(spec.header_rows + 1, ws.max_row - spec.header_rows)

        if spec.header_rows == 2:
            # Row 1 is intentionally retained to distinguish identical labels
            # belonging to different timepoints.
            for col, group in enumerate(spec.groups, 1):
                ws.cell(1, col).value = group or None
            for col, header in enumerate(spec.output_headers, 1):
                ws.cell(2, col).value = header
        else:
            for col, header in enumerate(spec.output_headers, 1):
                ws.cell(1, col).value = header

        frame = sheet_frames.get(sheet_name, pd.DataFrame())
        start_row = spec.header_rows + 1
        for row_offset, row in enumerate(frame.itertuples(index=False, name=None)):
            for col_offset, value in enumerate(row, 1):
                ws.cell(start_row + row_offset, col_offset).value = excel_value(value)

    wb.save(output_path)

def build_all_dataset_from_link(
    link_path,
    raw_patient_path,
    output_path,
    log_path,
    drugs_dict_path,
    com_dict_path,
    interact_dict_path,
    template_path,
):
    link = read_link(link_path)
    template = OutputTemplate(Path(template_path), link)
    dictionaries = DictionaryCache(Path(drugs_dict_path), Path(com_dict_path), Path(interact_dict_path))
    engine = ConverterEngine(dictionaries)
    patients = read_patient_sheets(raw_patient_path)

    all_sheet_frames = {name: [] for name in template.sheet_order}
    logs = []

    for patient_id, (patient_sheet_name, patient_df) in enumerate(patients.items(), 1):
        expansion_frames, patient_logs = execute_patient(
            patient_id, patient_sheet_name, patient_df, link, engine
        )
        logs.extend(patient_logs)

        for sheet_name in template.sheet_order:
            frame = make_patient_sheet_frame(
                patient_id,
                template.sheets[sheet_name],
                link,
                expansion_frames,
            )
            all_sheet_frames[sheet_name].append(frame)

    combined_sheets = {
        name: (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame())
        for name, frames in all_sheet_frames.items()
    }

    write_template_output(
        template_path=template_path,
        output_path=output_path,
        template=template,
        sheet_frames=combined_sheets,
    )

    log_df = pd.DataFrame(logs)
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(log_path, engine='openpyxl') as writer:
            log_df.to_excel(writer, sheet_name='mapping_log', index=False)
            if not log_df.empty:
                (
                    log_df['status']
                    .value_counts(dropna=False)
                    .rename_axis('status')
                    .reset_index(name='frequency')
                    .to_excel(writer, sheet_name='status_summary', index=False)
                )

    return combined_sheets, log_df


# ============================================================
BASE_DIR = Path('dati')
LINK_PATH = BASE_DIR / 'final files' / 'Link.xlsx'
TEMPLATE_PATH = BASE_DIR / 'working files' / 'dataset_finale_base.xlsx'
DICT_DIR = BASE_DIR / 'final files' / 'dict'
ROW_DATA_DIR = BASE_DIR / 'final files' / 'row_data'
OUTPUT_DIR = BASE_DIR / 'final files' / 'output'

DRUGS_DICT_PATH = DICT_DIR / 'Drugs_dict.xlsx'
COM_DICT_PATH = DICT_DIR / 'Com_dict.xlsx'
INTERACT_DICT_PATH = DICT_DIR / 'Interact_dict.xlsx'

PATIENT_FILE_NAME = 'pz_nefro_example.xls'

PATIENT_FILE_NAME = 'pz_nefro_example.xls'


def validate_manual_paths() -> None:
    required_files = [
        LINK_PATH,
        TEMPLATE_PATH,
        DRUGS_DICT_PATH,
        COM_DICT_PATH,
        INTERACT_DICT_PATH,
    ]
    missing = [str(path) for path in required_files if not path.is_file()]
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

    if not ROW_DATA_DIR.is_dir():
        raise FileNotFoundError(f'Patient-data folder not found: {ROW_DATA_DIR}')

def get_patient_files() -> list[Path]:
    if PATIENT_FILE_NAME:
        patient_path = ROW_DATA_DIR / PATIENT_FILE_NAME
        if not patient_path.is_file():
            raise FileNotFoundError(f'Patient workbook not found: {patient_path}')
        if patient_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            raise ValueError(f'Unsupported patient format: {patient_path.suffix}')
        return [patient_path]

    patient_files = sorted(
        path
        for path in ROW_DATA_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
        and not path.name.startswith('~$')
    )
    if not patient_files:
        raise FileNotFoundError(
            f'No .xls, .xlsx, or .ods patient workbooks found in {ROW_DATA_DIR}'
        )
    return patient_files

def convert_patient_file(patient_path: Path):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f'{patient_path.stem}_mapped.xlsx'
    log_path = OUTPUT_DIR / f'{patient_path.stem}_mapped_log.xlsx'

    sheet_frames, log_df = build_all_dataset_from_link(
        link_path=LINK_PATH,
        raw_patient_path=patient_path,
        output_path=output_path,
        log_path=log_path,
        drugs_dict_path=DRUGS_DICT_PATH,
        com_dict_path=COM_DICT_PATH,
        interact_dict_path=INTERACT_DICT_PATH,
        template_path=TEMPLATE_PATH,
    )

    print(f'\nCompleted: {patient_path.name}')
    print(f'Output: {output_path}')
    print(f'Log: {log_path}')
    print('Output rows per sheet:')
    for sheet_name, frame in sheet_frames.items():
        print(f'  - {sheet_name}: {len(frame)}')

    if not log_df.empty and 'status' in log_df.columns:
        print('\nMapping status frequencies:')
        print(log_df['status'].value_counts(dropna=False).to_string())

    return sheet_frames, log_df

def main() -> None:
    validate_manual_paths()
    patient_files = get_patient_files()

    print(f'LINK: {LINK_PATH}')
    print(f'Output template: {TEMPLATE_PATH}')
    print(f'Drugs dictionary: {DRUGS_DICT_PATH}')
    print(f'Comorbidity dictionary: {COM_DICT_PATH}')
    print(f'Interaction dictionary: {INTERACT_DICT_PATH}')
    print(f'Patient workbooks to process: {len(patient_files)}')

    for patient_path in patient_files:
        convert_patient_file(patient_path)


if __name__ == '__main__':
    main()



LINK: dati\final files\Link.xlsx
Output template: dati\working files\dataset_finale_base.xlsx
Drugs dictionary: dati\final files\dict\Drugs_dict.xlsx
Comorbidity dictionary: dati\final files\dict\Com_dict.xlsx
Interaction dictionary: dati\final files\dict\Interact_dict.xlsx
Patient workbooks to process: 1


C:\Users\User\AppData\Local\Temp\ipykernel_45616\701988818.py:429: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
C:\Users\User\AppData\Local\Temp\ipykernel_45616\701988818.py:429: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
C:\Users\User\AppData\Local\Temp\ipykernel_45616\701988818.py:429: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
C:\Users\User\AppData\Local\Temp\ipykernel_45616\701988818.py:429: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
C:\Users\User\AppData\Local\Temp\ipy


Completed: pz_nefro_example.xls
Output: dati\final files\output\pz_nefro_example_mapped.xlsx
Log: dati\final files\output\pz_nefro_example_mapped_log.xlsx
Output rows per sheet:
  - GENERALE: 3
  - COMORBIDITA: 24
  - TERAPIA PRE MR E MEDICATION REV: 52
  - TERAPIA POST MR (X): 3
  - TERAPIA FU 1 (X): 3
  - TERAPIA FU 2 (X): 3
  - INTERAZIONI: 17
  - ADR: 3

Mapping status frequencies:
status
mapped                               74
source_not_found_in_patient_sheet     7
source_found_but_empty                3


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
